# 🤖 Notebook 02 — Model Training: scikit-learn + MLflow

**Goal:** Train Logistic Regression and Random Forest classifiers. Track all experiments with MLflow.

> **Run time:** ~5 min

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, average_precision_score)
from sklearn.pipeline import Pipeline

# Load feature store
df = spark.table('silver_credit_risk_features').toPandas()
print(f'Loaded {len(df)} records')
print(f'Default rate: {df["IsDefault"].mean():.1%}')

## Step 1 — Prepare Features

In [ ]:
# Encode categoricals
cat_cols = ['AccountType','Branch','Status','LoanPurpose','EmploymentType','HomeOwnership']
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

feature_cols = [
    'CreditScore','AnnualIncome','LoanAmount','EmploymentYears',
    'DebtToIncomeRatio','NumOpenAccounts','NumDelinquencies',
    'MonthsSinceLastDelinquency','LoanTermMonths',
    'TotalTransactions','TotalLoanAmount','AvgTransactionAmount',
    'NumLoans','MaxTransactionAmount',
    'AccountType_enc','LoanPurpose_enc','EmploymentType_enc','HomeOwnership_enc'
]

X = df[feature_cols].fillna(0)
y = df['IsDefault'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Test default rate: {y_test.mean():.1%}')

## Step 2 — Logistic Regression

In [ ]:
mlflow.set_experiment('CreditRiskScoring')

with mlflow.start_run(run_name='LogisticRegression'):
    pipeline_lr = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(max_iter=1000, C=0.1, random_state=42))
    ])
    pipeline_lr.fit(X_train, y_train)

    y_prob_lr = pipeline_lr.predict_proba(X_test)[:, 1]
    auc_lr    = roc_auc_score(y_test, y_prob_lr)
    ap_lr     = average_precision_score(y_test, y_prob_lr)

    mlflow.log_param('model_type', 'LogisticRegression')
    mlflow.log_param('C', 0.1)
    mlflow.log_metric('auc_roc', auc_lr)
    mlflow.log_metric('avg_precision', ap_lr)
    mlflow.sklearn.log_model(pipeline_lr, 'logistic_regression')

    print(f'Logistic Regression  |  AUC-ROC: {auc_lr:.4f}  |  Avg Precision: {ap_lr:.4f}')
    print(classification_report(y_test, pipeline_lr.predict(X_test), target_names=['No Default','Default']))

## Step 3 — Random Forest

In [ ]:
with mlflow.start_run(run_name='RandomForest'):
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10,
        min_samples_leaf=5, random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)

    y_prob_rf = rf.predict_proba(X_test)[:, 1]
    auc_rf    = roc_auc_score(y_test, y_prob_rf)
    ap_rf     = average_precision_score(y_test, y_prob_rf)

    mlflow.log_param('model_type', 'RandomForest')
    mlflow.log_param('n_estimators', 200)
    mlflow.log_param('max_depth', 10)
    mlflow.log_metric('auc_roc', auc_rf)
    mlflow.log_metric('avg_precision', ap_rf)
    mlflow.sklearn.log_model(rf, 'random_forest')

    print(f'Random Forest        |  AUC-ROC: {auc_rf:.4f}  |  Avg Precision: {ap_rf:.4f}')
    print(classification_report(y_test, rf.predict(X_test), target_names=['No Default','Default']))

## Step 4 — View MLflow Experiments

In [ ]:
# View all runs in this experiment
runs = mlflow.search_runs(experiment_names=['CreditRiskScoring'])
print(runs[['tags.mlflow.runName','metrics.auc_roc','metrics.avg_precision']].sort_values('metrics.auc_roc', ascending=False))